# 08 — Feature importance
Importâncias do XGBoost treinado em 2020–2023. Agrupamentos de One-Hot podem favorecer variáveis com alta cardinalidade.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import joblib
ROOT=Path.cwd(); OUT=ROOT/"outputs"; DATA=ROOT/"data"/"processed"
SILVER=DATA/"prf_acidentes_silver_2020_2025.parquet"
for p in ["modelos","interpretabilidade","modelo_final"]:
    (OUT/p).mkdir(parents=True,exist_ok=True)
df=pd.read_parquet(SILVER)
df["br_categoria"]=pd.to_numeric(df.br,errors="coerce").astype("Int64").astype("string").fillna("Não informado")
flags=sorted(c for c in df if c.startswith("tracado_") and c!="tracado_via")
categoricas=["uf","br_categoria","dia_semana","fase_dia","sentido_via","condicao_metereologica","tipo_pista","uso_solo","tipo_acidente"]
numericas=["km","mes","hora","fim_semana","qtd_caracteristicas_tracado","veiculos"]+flags
features=categoricas+numericas
proibidas={"pessoas","mortos","feridos_graves","feridos_leves","feridos","ilesos","ignorados","classificacao_acidente","acidente_fatal"}
assert set(features).isdisjoint(proibidas)
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
def pre(cat=categoricas,num=numericas):
    try: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse_output=True)
    except TypeError: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse=True)
    return ColumnTransformer([("categoricas",enc,cat),("numericas",StandardScaler(),num)])
clf=joblib.load(OUT/"modelos"/"xgboost.joblib")
preproc=clf.named_steps["preprocessamento"]
modelo=clf.named_steps["modelo"]
imp=pd.DataFrame({"feature":preproc.get_feature_names_out(),"importance":modelo.feature_importances_}).sort_values("importance",ascending=False)
display(imp.head(25))
imp.to_csv(OUT/"interpretabilidade"/"feature_importance_xgboost.csv",index=False)
def origem(s):
    s=s.replace("categoricas__","").replace("numericas__","")
    for c in sorted(categoricas,key=len,reverse=True):
        if s==c or s.startswith(c+"_"):return c
    return s
imp["variavel_original"]=imp.feature.map(origem)
group=imp.groupby("variavel_original",as_index=False).importance.sum().sort_values("importance",ascending=False)
group["percentual_importancia"]=group.importance/group.importance.sum()*100
display(group)
group.to_csv(OUT/"interpretabilidade"/"importancia_variaveis_agrupadas.csv",index=False)
pd.DataFrame({"parametro":["modelo","threshold","criterio_threshold","periodo_treino","periodo_validacao","periodo_teste_final"],
"valor":["XGBoost",".28","maior F2-score","2020-2023","2024","2025"]}).to_csv(OUT/"modelos"/"configuracao_modelo_final.csv",index=False)